# Step 2 and Step 3 Walkthrough: Tracking-First Table + Smoothed Possession

This notebook explains the Step 2 master-join pipeline for one match, checks the final all-match master join table, and then runs Step 3 to smooth possession over time.

The project logic is tracking-first:

- tracking frames are the main table
- events are joined onto tracking frames by match-clock time only
- every tracking row remains in the final table
- rows with no matched event get `event_label = "no event"` and `event_type_name = "no event"`
- matched event columns are kept with an `event_` prefix
- Step 2 event labels are limited to `PASS`, `BALL TOUCH`, `AERIAL`, `TACKLE`, `BALL RECOVERY`, `FOUL`, and `TAKEON`

Join keys:

- event key: `period_id + min + sec + milisec`, converted to `event_match_clock_seconds`
- tracking key: `period_id + match_clock + frame_position_inside_second / FPS`, stored as `tracking_match_clock_seconds`
- join method: nearest tracking match-clock frame within the same period

Step 3 then starts from the all-match master join table and creates smoothed possession assignment and possession-change columns.

Default example match: `679026`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from driblab.config import CONFIG_PATH, RAW_DATA_DIR
from driblab.config import POSSESSION_SEQUENCE_DATA_DIR
from driblab.etl.pipeline import load_event_file, load_tracking_file
from driblab.etl.master_join import (
    add_event_match_clock_seconds,
    add_event_attack_directions,
    add_possession_features,
    build_live_frame_features,
    build_model_base_frame_table,
    build_player_features,
    build_tracking_frame_table,
    filter_step2_events,
    infer_attack_directions,
    match_events_to_frames,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

MATCH_ID = "679026"
JOIN_TOLERANCE_SEC = 0.5
STEP2_EVENT_TYPES = (
    "PASS",
    "BALL TOUCH",
    "AERIAL",
    "TACKLE",
    "BALL RECOVERY",
    "FOUL",
    "TAKEON",
)

from driblab.features.match_splits import add_data_split_column
from driblab.features.match_splits import load_match_splits
from driblab.features.match_splits import summarize_splits
from driblab.features.possession_sequence import Step3Config, run_step3

RAW_DATA_DIR


## 1. Load One Match

Load the raw events JSON and tracking JSONL for one match. The tracking file has a header row, then one JSON object per frame.

In [ ]:
events_path = RAW_DATA_DIR / f"{MATCH_ID}_events.json"
tracking_path = RAW_DATA_DIR / f"{MATCH_ID}_tracking_data.jsonl"

df_events_raw = pd.json_normalize(load_event_file(events_path))
df_events = filter_step2_events(df_events_raw, STEP2_EVENT_TYPES)
tracking_header, tracking_frames = load_tracking_file(tracking_path)

print(f"Match: {MATCH_ID}")
print(f"Raw events: {len(df_events_raw):,}")
print(f"Step 2 selected events: {len(df_events):,}")
print(f"Tracking frames: {len(tracking_frames):,}")
print(f"FPS: {tracking_header.get('FPS')}")
print(tracking_header.get("match_data"))

df_events.head()

## 2. Build Tracking Match-Clock Time

Tracking is the base table. Each tracking row has raw match clock as `match_clock = [minute, second]`, but the data is 10 Hz, so several frames share the same integer second.

Step 2 creates one tracking join key:

`tracking_match_clock_seconds = match_clock_min * 60 + match_clock_sec + frame_position_inside_second / FPS`

Example at 10 FPS: if the tracking match clock is 2 seconds and the frame is the 4th frame inside that second, then `tracking_match_clock_seconds = 2 + 4 / 10 = 2.4`.

The raw nested player lists are not stored in the final Parquet table for now. The final table keeps flattened frame columns plus engineered player aggregates.


In [ ]:
frame_df, video_clock_diagnostics = build_tracking_frame_table(
    tracking_header,
    tracking_frames,
)

tracking_time_cols = [
    "frame_id",
    "period_id",
    "match_clock_min",
    "match_clock_sec",
    "tracking_match_clock_seconds",
    "video_timestamp",
    "cam_present",
    "ball_x_raw",
    "ball_y_raw",
    "ball_z_raw",
]

display(frame_df[tracking_time_cols].head(15))

print("Video-clock diagnostics only, not used for event join:")
display(video_clock_diagnostics)

## 3. Build Event Match-Clock Time

Events already use match clock. We convert `min`, `sec`, and `milisec` into seconds.

`event_match_clock_seconds = min * 60 + sec + milisec / 1000`

In [ ]:
df_events_timed = add_event_match_clock_seconds(df_events)

event_time_cols = [
    "event.id",
    "event.event_type_name",
    "period_id",
    "min",
    "sec",
    "milisec",
    "event_match_clock_seconds",
    "team.team_name",
    "player.player_name",
]

display(df_events_timed[event_time_cols].head(15))

print("Added one event join column: event_match_clock_seconds")

## 4. Sync Events to Tracking by Match Clock

This is the event-to-tracking timestamp sync, and it is also the drift-correction step. For each period separately, Step 2 uses `pd.merge_asof`:

- left table: events
- right table: tracking frames
- left key: `event_match_clock_seconds`
- right key: `tracking_match_clock_seconds`
- direction: `nearest`
- tolerance: `0.5` seconds

The join QA field is:

`event_match_clock_join_error_sec = event_match_clock_seconds - event_matched_tracking_match_clock_seconds`

A value near zero means the selected tracking frame is very close to the event time.


In [ ]:
df_events_matched = match_events_to_frames(
    df_events_timed,
    frame_df,
    tolerance_sec=JOIN_TOLERANCE_SEC,
)

join_cols = [
    "event.id",
    "event.event_type_name",
    "period_id",
    "event_match_clock_seconds",
    "frame_id",
    "matched_match_clock_min",
    "matched_match_clock_sec",
    "matched_tracking_match_clock_seconds",
    "match_clock_join_error_sec",
    "matched_cam_present",
    "matched_ball_x_raw",
    "matched_ball_y_raw",
]

display(df_events_matched[join_cols].head(20))

matched = df_events_matched.dropna(subset=["frame_id"])
print(f"Matched events: {len(matched):,} / {len(df_events_matched):,}")
print(f"Median absolute match-clock error: {matched['match_clock_join_error_sec'].abs().median():.3f}s")
print(f"95th percentile absolute match-clock error: {matched['match_clock_join_error_sec'].abs().quantile(0.95):.3f}s")
print(f"Max absolute match-clock error: {matched['match_clock_join_error_sec'].abs().max():.3f}s")

## 5. Inspect One Event and Nearby Tracking Frames

This lets you visually check the exact event time and the nearby tracking rows.

In [ ]:
example_event = matched.iloc[0]
period_id = int(example_event["period_id"])
frame_id = int(example_event["frame_id"])

print("Matched event:")
display(example_event[join_cols + ["team.team_name", "player.player_name"]].to_frame("value"))

nearby_frames = frame_df[
    (frame_df["period_id"] == period_id)
    & frame_df["frame_id"].between(frame_id - 5, frame_id + 5)
][tracking_time_cols]

print("Nearby tracking rows:")
display(nearby_frames)

## 6. Infer Attacking Direction

After time matching, event coordinates are kept on their provider `0-100` field scale. Tracking x/y is later converted into attacking-perspective columns using the event, possession, or nearest-team reference.


In [ ]:
directions, direction_df = infer_attack_directions(
    df_events_matched,
    tracking_header,
    tolerance_sec=0.30,
)
df_events_aligned = add_event_attack_directions(df_events_matched, directions)

display(direction_df)

coord_cols = [
    "event.event_type_name",
    "team.team_name",
    "period_id",
    "attack_direction",
    "x",
    "y",
    "frame_id",
    "matched_tracking_match_clock_seconds",
]
display(df_events_aligned[coord_cols].head(20))


## 7. Build the Tracking-First Training Table for One Match

The final master join table keeps one row per tracking frame. Event columns are joined onto those tracking rows when one of the selected Step 2 event types matched that frame.

Important fields:

- `event_label`: event name, or `no event`
- `event_type_name`: original event type name, or `no event`
- `event_count_at_frame`: number of events matched to that tracking frame
- `is_event_frame`: boolean flag for whether the tracking row has an event
- `event_*`: original event columns and Step 2 event-side join/coordinate fields

Ball speed and acceleration are calculated from tracking-derived ball positions before the saved x/y columns are normalized to the `0-100` field scale. A consecutive frame means the next tracking row in the same `period_id` after sorting by `video_timestamp`; it is not assumed to be `frame_id + 1`.

`dt_sec` is the elapsed time from the previous tracking frame in the same period. Speed and acceleration are only calculated for valid timing gaps where `0 < dt_sec <= max_speed_dt_sec` and both rows have camera data; this notebook uses `max_speed_dt_sec=0.50`. Larger gaps are left missing to avoid artificial movement spikes.

For valid gaps, `ball_speed_mps = sqrt(dx^2 + dy^2 + dz^2) / dt_sec`, and `ball_acceleration_mps2 = (current ball_speed_mps - previous ball_speed_mps) / dt_sec`. This is acceleration based on change in speed, not a full x/y/z acceleration vector.


In [ ]:
live_frame_features = build_live_frame_features(
    frame_df,
    max_ball_gap_frames=10,
    max_speed_dt_sec=0.50,
)
player_features = build_player_features(
    tracking_header,
    tracking_frames,
    live_frame_features,
    max_speed_dt_sec=0.50,
)
frame_features = add_possession_features(
    live_frame_features,
    player_features,
    possession_distance_m=2.0,
    possession_max_ball_speed_mps=12.0,
)
master_join_one_match = build_model_base_frame_table(
    MATCH_ID,
    frame_features,
    player_features,
    df_events_aligned,
    raw_event_columns=list(df_events.columns),
    directions=directions,
)

master_join_overview = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "event-prefixed columns",
            "no-event rows",
            "event rows",
        ],
        "value": [
            len(master_join_one_match),
            len(master_join_one_match.columns),
            sum(col.startswith("event_") for col in master_join_one_match.columns),
            master_join_one_match["event_label"].eq("no event").sum(),
            master_join_one_match["is_event_frame"].sum(),
        ],
    }
)
display(master_join_overview)

coordinate_check_cols = [
    'ball_x_raw',
    'ball_y_raw',
    'ball_x',
    'ball_y',
    'event_x',
    'event_y',
    'ball_x_attacking',
    'ball_y_attacking',
]
coordinate_check = []
for col in coordinate_check_cols:
    if col in master_join_one_match.columns:
        values = pd.to_numeric(master_join_one_match[col], errors='coerce').dropna()
        coordinate_check.append({'column': col, 'min': values.min(), 'max': values.max()})
display(pd.DataFrame(coordinate_check))

master_join_cols = [
    "match_id",
    "frame_id",
    "period_id",
    "match_clock_min",
    "match_clock_sec",
    "tracking_match_clock_seconds",
    "dt_sec",
    "ball_x",
    "ball_y",
    "ball_speed_mps",
    "ball_acceleration_mps2",
    "has_possession",
    "event_label",
    "event_type_name",
    "event_team_name",
    "event_player_name",
    "event_match_clock_seconds",
    "event_matched_tracking_match_clock_seconds",
    "event_match_clock_join_error_sec",
    "event_count_at_frame",
    "is_event_frame",
]

display(master_join_one_match[master_join_cols].head(20))
display(master_join_one_match.loc[master_join_one_match["is_event_frame"], master_join_cols].head(20))


## 8. Check the Final All-Match Output

This cell reads the generated final Parquet table and summarizes the table that should be used as the basis for modelling:

`data/processed/model_base/master_join_table.parquet`

It confirms the number of rows, number of columns, event-column count, and how many tracking rows are labelled `no event`.


In [ ]:
master_join_table_path = PROJECT_ROOT / "data" / "processed" / "model_base" / "master_join_table.parquet"
summary_path = PROJECT_ROOT / "data" / "processed" / "model_base" / "master_join_summary.csv"

master_join_table_header = pq.read_schema(master_join_table_path).names
summary = pd.read_csv(summary_path)

event_label_sample = pd.read_parquet(
    master_join_table_path,
    columns=["match_id", "frame_id", "event_label", "event_type_name", "is_event_frame"],
)
rows = len(event_label_sample)
raw_tracking_rows = sum(
    sum(1 for _ in path.open()) - 1
    for path in RAW_DATA_DIR.glob("*_tracking_data.jsonl")
)
unique_match_frame_rows = event_label_sample.groupby(["match_id", "frame_id"]).ngroups
no_event_rows = int(event_label_sample["event_label"].eq("no event").sum())
event_rows = int(event_label_sample["is_event_frame"].sum())
label_counts = event_label_sample["event_label"].value_counts(dropna=False).to_dict()
allowed_labels = set(STEP2_EVENT_TYPES) | {"no event"}
observed_atomic_labels = {
    part.strip()
    for value in event_label_sample["event_label"].dropna().astype(str)
    for part in value.split("|")
}
unexpected_labels = sorted(observed_atomic_labels - allowed_labels)
removed_player_speed_columns = [
    "mean_player_speed_mps",
    "max_player_speed_mps",
    "mean_reliable_player_speed_mps",
    "max_reliable_player_speed_mps",
]
present_removed_columns = [
    col for col in removed_player_speed_columns if col in master_join_table_header
]

final_table_overview = pd.DataFrame(
    {
        "metric": [
            "matches",
            "raw tracking rows",
            "rows",
            "unique (match_id, frame_id) rows",
            "columns",
            "event-prefixed columns",
            "non-event tracking/feature columns",
            "rows labelled no event",
            "rows with at least one event",
            "mean median absolute join error seconds",
            "mean p95 absolute join error seconds",
        ],
        "value": [
            summary["match_id"].nunique(),
            raw_tracking_rows,
            rows,
            unique_match_frame_rows,
            len(master_join_table_header),
            sum(col.startswith("event_") for col in master_join_table_header),
            len(master_join_table_header) - sum(col.startswith("event_") for col in master_join_table_header),
            no_event_rows,
            event_rows,
            round(summary["median_abs_match_clock_join_error_sec"].mean(), 3),
            round(summary["p95_abs_match_clock_join_error_sec"].mean(), 3),
        ],
    }
)
sanity_checks = pd.DataFrame(
    [
        {
            "check": "master rows equal raw tracking rows",
            "passed": rows == raw_tracking_rows,
            "detail": f"master={rows:,}; raw_tracking={raw_tracking_rows:,}",
        },
        {
            "check": "one row per (match_id, frame_id)",
            "passed": rows == unique_match_frame_rows,
            "detail": f"rows={rows:,}; unique={unique_match_frame_rows:,}",
        },
        {
            "check": "event labels limited to Step 2 selected types",
            "passed": not unexpected_labels,
            "detail": ", ".join(unexpected_labels) if unexpected_labels else "OK",
        },
        {
            "check": "player speed aggregate columns removed",
            "passed": not present_removed_columns,
            "detail": ", ".join(present_removed_columns) if present_removed_columns else "OK",
        },
    ]
)

display(final_table_overview)
display(sanity_checks)
display(pd.DataFrame(sorted(label_counts.items(), key=lambda item: item[1], reverse=True)[:12], columns=["event_label", "rows"]))
display(pd.DataFrame({"columns": master_join_table_header}))


## 9. Step 3: Smooth Possession Over Time

Step 3 follows the project instruction: assign the ball to the nearest player, smooth that assignment over time, and track when possession changes.

This belongs after Step 2 because Step 3 uses the Step 2 master join table as its only input. It does not change the event-to-tracking join. It adds possession columns that describe who has the ball after smoothing and when the ball changes player or team.


## 10. Apply Match-Level Splits Before Inspecting Step 3

The split still happens by complete `match_id`, never by individual frame. This prevents adjacent 10 Hz frames from the same match leaking between train, validation, and test.

The next cell loads only the columns needed to confirm the split and possession inputs before running Step 3.


In [ ]:
splits_path = CONFIG_PATH
splits = load_match_splits(splits_path)

step3_preview_cols = [
    'match_id',
    'frame_id',
    'period_id',
    'tracking_match_clock_seconds',
    'event_label',
    'has_possession',
    'possessing_team_id',
    'possessing_player_id',
]
step3_input_preview = pd.read_parquet(
    master_join_table_path,
    columns=step3_preview_cols,
)
step3_input_with_split = add_data_split_column(step3_input_preview, splits)

print(f'Splits path: {splits_path}')
for split_name in ['train', 'validation', 'test']:
    print(f'{split_name}: {splits[split_name]}')

summarize_splits(step3_input_with_split)


## 11. Run Step 3

The next cell runs the same Step 3 function used by `python main.py step3`.

It writes:

- `data/processed/possession_sequence/possession_sequence_table.parquet`
- `data/processed/possession_sequence/possession_sequence_summary.csv`
- `data/processed/possession_sequence/possession_sequence_metadata.json`

The smoothing parameters are:

- `max_gap_frames=3`: fill very short no-possession gaps when the same player has the ball before and after the gap
- `min_stable_frames=3`: remove very short possession flickers surrounded by the same possessor


In [ ]:
step3_result = run_step3(
    Step3Config(
        input_table=master_join_table_path,
        output_dir=POSSESSION_SEQUENCE_DATA_DIR,
        match_splits=splits_path,
        max_gap_frames=3,
        min_stable_frames=3,
    )
)

possession_sequence = step3_result['table']
possession_sequence_summary = step3_result['summary']

print(step3_result['outputs'])
possession_sequence_summary.head(10)


## 12. Inspect the Smoothed Possession Columns

This cell compares the raw Step 2 nearest-player possession assignment with the Step 3 smoothed possession assignment.

Important columns:

- `raw_possession_key`: Step 2 assignment, built from raw `possessing_team_id` and `possessing_player_id`
- `smoothed_possession_*`: Step 3 assignment after smoothing
- `possession_*_change`: whether the smoothed possession changed player or team
- `possession_change_type`: readable change category


In [ ]:
step3_columns = [
    'match_id',
    'period_id',
    'frame_id',
    'tracking_match_clock_seconds',
    'event_label',
    'nearest_player_name',
    'nearest_player_distance_to_ball_m',
    'raw_possession_key',
    'smoothed_possession_team_name',
    'smoothed_possession_player_name',
    'smoothed_possession_change',
    'possession_team_change',
    'possession_player_change',
    'possession_change_type',
    'data_split',
]

possession_sequence[step3_columns].head(20)


## 13. Inspect Possession Changes

These possession-change rows are the skeleton for many later event rules. For example, a same-team player change can become a pass candidate, while an opponent team change can become an interception or tackle candidate.


In [ ]:
change_rows = possession_sequence[
    possession_sequence['smoothed_possession_change']
].copy()

print(f'Possession changes: {len(change_rows):,}')
change_rows[
    [
        'match_id',
        'period_id',
        'frame_id',
        'tracking_match_clock_seconds',
        'previous_smoothed_possession_team_id',
        'smoothed_possession_team_id',
        'previous_smoothed_possession_player_id',
        'smoothed_possession_player_id',
        'possession_change_type',
        'event_label',
        'ball_speed_mps',
    ]
].head(20)
